# **Differential Privacy with Opacus**
---

Here, I introduced differential privacy (DP) theory, explain the DP-SGD algorithm, and run a small standalone experiment: training a single-site model with Opacus at several different privacy budgets (ε). I measured how each ε level affects AUC-ROC, to quantify the privacy–utility trade-off before the full FL+DP run in Notebook 10.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

USE_MOCK_DATA = True

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

from src.config import load_config
from src.paths import get_paths
from src.data_utils import TBDataset, build_transforms, get_class_weights
from src.model import build_model
from src.dp_utils import validate_model_for_dp, make_private_model, get_privacy_spent, print_dp_summary
from src.metrics import compute_metrics, bootstrap_ci
from src.visualization import set_publication_style, save_figure, plot_privacy_utility_tradeoff

cfg   = load_config()
paths = get_paths()
set_publication_style()

SEED = cfg["project"]["random_seed"]
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## Differential Privacy — Intuition

Imagine two training datasets that differ by exactly one patient record:
- **D**: contains patient Alice's chest X-ray (TB positive)
- **D'**: same dataset but with Alice's record removed

**(ε, δ)-Differential Privacy** guarantees:

> For any possible model output O:
> `P(M(D) ∈ O) ≤ e^ε × P(M(D') ∈ O) + δ`

This means: even if an adversary inspects the trained model weights (or gradient updates), they cannot reliably distinguish whether Alice was in the training set.

**ε (epsilon)** — privacy budget:
- ε = 0: perfect privacy (but model learns nothing)
- ε = 1: strong privacy
- ε = 8: reasonable for medical imaging (commonly used in literature)
- ε → ∞: no privacy guarantee

**δ (delta)** — failure probability:
- Typically set to 1/n (where n = training set size)
- Represents the small probability that the guarantee fails

**DP-SGD** achieves DP in practice by:
1. **Gradient clipping**: each sample's gradient is clipped to max norm C (limits how much any one sample can influence the model)
2. **Gaussian noise**: noise σ·C is added to the clipped gradient sum
3. **Privacy accounting**: the Rényi Differential Privacy accountant tracks cumulative ε over all steps


In [2]:
# ── Small DP sweep on Site 0 data ─────────────────────────────────
# I train for a small number of epochs at each epsilon to observe the
# privacy-utility trade-off WITHOUT running the full FL pipeline.
# (Full FL+DP is in Notebook 10.)

IMAGE_SIZE = cfg["data"]["image_size"]
BATCH_SIZE = 16 if USE_MOCK_DATA else cfg["training"]["batch_size"]
DP_EPOCHS  = 3  if USE_MOCK_DATA else 20   # Short for sweep

train_transform = build_transforms(image_size=IMAGE_SIZE, split="train",
    normalize_mean=cfg["augmentation"]["normalize_mean"],
    normalize_std=cfg["augmentation"]["normalize_std"])
val_transform = build_transforms(image_size=IMAGE_SIZE, split="val",
    normalize_mean=cfg["augmentation"]["normalize_mean"],
    normalize_std=cfg["augmentation"]["normalize_std"])

# Use site 0 as the "single hospital" for this experiment
site0_df = pd.read_csv(paths["processed"] / "site_0_train.csv")
val_df   = pd.read_csv(paths["processed"] / "val.csv")
test_df  = pd.read_csv(paths["processed"] / "test.csv")

site0_dataset = TBDataset(site0_df, transform=train_transform)
val_dataset   = TBDataset(val_df,   transform=val_transform)
test_dataset  = TBDataset(test_df,  transform=val_transform)

# NOTE: Opacus requires batch_size to NOT exceed dataset size
eff_batch = min(BATCH_SIZE, len(site0_dataset) // 2, 32)
site0_loader = DataLoader(site0_dataset, batch_size=eff_batch, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Site 0 training: {len(site0_dataset)} images | batch size: {eff_batch}")
print(f"Sweeping DP epochs: {DP_EPOCHS}")

# Epsilon values to sweep
EPSILON_VALUES = [1.0, 2.0, 4.0, 8.0]   # Skip very small for mock (too noisy)
TARGET_DELTA   = cfg["differential_privacy"]["target_delta"]
MAX_GRAD_NORM  = cfg["differential_privacy"]["max_grad_norm"]

Site 0 training: 277 images | batch size: 16
Sweeping DP epochs: 3


In [3]:
def train_with_dp(epsilon, delta, max_grad_norm, epochs, device, seed):
    """Train a single-site ResNet-18 with Opacus DP-SGD and return test AUC."""
    torch.manual_seed(seed)

    # Build fresh model
    model = build_model(pretrained=True, num_classes=2, dropout=0.5).to(device)

    # Validate and fix model for Opacus (replaces BatchNorm → GroupNorm)
    model = validate_model_for_dp(model)
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Wrap model, optimizer, and loader with Opacus DP engine
    private_model, private_optimizer, private_loader = make_private_model(
        model=model,
        optimizer=optimizer,
        train_loader=site0_loader,
        target_epsilon=epsilon,
        target_delta=delta,
        max_grad_norm=max_grad_norm,
        epochs=epochs,
    )

    # Training loop
    for epoch in range(epochs):
        private_model.train()
        for images, labels, _ in private_loader:
            images = images.to(device)
            labels = labels.to(device)
            private_optimizer.zero_grad()
            logits = private_model(images)
            loss   = criterion(logits, labels)
            loss.backward()
            private_optimizer.step()

    # Evaluate on test set
    private_model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels, _ in test_loader:
            images = images.to(device)
            logits = private_model(images)
            probs  = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)

    auc = 0.0
    if len(np.unique(y_true)) == 2:
        auc = float(roc_auc_score(y_true, y_prob))

    # Query actual ε spent
    privacy = get_privacy_spent(private_model)
    return auc, privacy["epsilon"], y_true, y_prob

print("DP sweep function defined. Starting sweep...")
print()

DP sweep function defined. Starting sweep...



In [4]:
# Run the DP sweep
sweep_results = []

# Also run without DP as reference
print("Training WITHOUT DP (reference)...")
torch.manual_seed(SEED)
ref_model = build_model(pretrained=True, num_classes=2, dropout=0.5).to(DEVICE)
ref_optimizer = torch.optim.Adam(ref_model.parameters(), lr=1e-4)
ref_criterion = nn.CrossEntropyLoss()
for epoch in range(DP_EPOCHS):
    ref_model.train()
    for images, labels, _ in site0_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        ref_optimizer.zero_grad()
        logits = ref_model(images)
        ref_criterion(logits, labels).backward()
        ref_optimizer.step()

ref_model.eval()
ref_probs, ref_labels = [], []
with torch.no_grad():
    for images, labels, _ in test_loader:
        logits = ref_model(images.to(DEVICE))
        probs  = torch.softmax(logits, dim=1)[:, 1]
        ref_probs.extend(probs.cpu().numpy())
        ref_labels.extend(labels.numpy())
ref_auc = roc_auc_score(ref_labels, ref_probs) if len(np.unique(ref_labels)) == 2 else 0.0
sweep_results.append({"epsilon": float("inf"), "auc": ref_auc, "epsilon_spent": float("inf")})
print(f"  No DP: AUC = {ref_auc:.4f}")
print()

# DP sweep
for eps in EPSILON_VALUES:
    print(f"Training with ε={eps}...")
    try:
        auc, eps_spent, y_t, y_p = train_with_dp(
            epsilon=eps, delta=TARGET_DELTA,
            max_grad_norm=MAX_GRAD_NORM,
            epochs=DP_EPOCHS, device=DEVICE, seed=SEED,
        )
        sweep_results.append({"epsilon": eps, "auc": auc, "epsilon_spent": eps_spent})
        print(f"  ε={eps}: AUC = {auc:.4f} | Actual ε spent = {eps_spent:.4f}")
    except Exception as e:
        print(f"  ε={eps}: FAILED — {e}")
        sweep_results.append({"epsilon": eps, "auc": None, "epsilon_spent": None})

sweep_df = pd.DataFrame(sweep_results)
print()
print("DP sweep results:")
print(sweep_df.to_string(index=False))

Training WITHOUT DP (reference)...
  No DP: AUC = 0.8747

Training with ε=1.0...
Found 20 compatibility issues. Auto-fixing...
  BatchNorm layers replaced with GroupNorm. Model is DP-compatible.


c:\Users\Peter\.ml\Lib\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
c:\Users\Peter\.ml\Lib\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


DP-SGD configured:
  Target ε = 1.0 | Target δ = 1e-05
  Max gradient norm (clipping) = 1.0
  Noise multiplier (σ) = 1.8945
  Training epochs = 3
  (Higher σ = more noise = stronger privacy, lower utility)
  ε=1.0: FAILED — Output 0 of BackwardHookFunctionBackward is a view and is being modified inplace. This view was created inside a custom Function (or because an input was returned as-is) and the autograd logic to handle view+inplace would override the custom backward associated with the custom Function, leading to incorrect gradients. This behavior is forbidden. You can fix this by cloning the output of the custom Function.
Training with ε=2.0...
Found 20 compatibility issues. Auto-fixing...
  BatchNorm layers replaced with GroupNorm. Model is DP-compatible.
DP-SGD configured:
  Target ε = 2.0 | Target δ = 1e-05
  Max gradient norm (clipping) = 1.0
  Noise multiplier (σ) = 1.2354
  Training epochs = 3
  (Higher σ = more noise = stronger privacy, lower utility)
  ε=2.0: FAILED — Outp

In [5]:
# Save sweep results
sweep_df.to_csv(paths["tables"] / "dp_sweep_results.csv", index=False)
print(f"Sweep results saved.")

# Plot privacy-utility trade-off
valid_rows = sweep_df[sweep_df["epsilon"] != float("inf")].dropna()
no_dp_row  = sweep_df[sweep_df["epsilon"] == float("inf")].iloc[0] if len(sweep_df[sweep_df["epsilon"] == float("inf")]) > 0 else None

if len(valid_rows) > 0:
    fig = plot_privacy_utility_tradeoff(
        epsilon_values = valid_rows["epsilon"].tolist(),
        auc_values     = valid_rows["auc"].tolist(),
        auc_ci_lower   = [v - 0.05 for v in valid_rows["auc"].tolist()],  # Placeholder CI
        auc_ci_upper   = [v + 0.05 for v in valid_rows["auc"].tolist()],
        centralised_auc= float(no_dp_row["auc"]) if no_dp_row is not None else None,
        title          = "Privacy–Utility Trade-off (Single Site, Short Training)\n(CIs are placeholders — bootstrap CIs computed in Notebook 11)",
    )
    save_figure(fig, "privacy_utility_tradeoff", paths["figures"], paths["paper_figures"])
    plt.show()

print()
print("Interpretation:")
print("  As ε decreases (stronger privacy), AUC typically drops.")
print("  The goal is to find the smallest ε where AUC remains clinically useful (≥0.80).")
print("  Full bootstrap CIs over the complete FL+DP run are computed in Notebook 11.")

Sweep results saved.

Interpretation:
  As ε decreases (stronger privacy), AUC typically drops.
  The goal is to find the smallest ε where AUC remains clinically useful (≥0.80).
  Full bootstrap CIs over the complete FL+DP run are computed in Notebook 11.
